In [1]:
import numpy as np
import pandas as pd
from web3 import Web3
import time
import matplotlib.pyplot as plt

In [2]:
NODE_URL = 'https://rpc.berachain.com'
w3 = Web3(Web3.HTTPProvider(NODE_URL))

In [3]:
def create_contract_web3(address, abi_path):
    with open(abi_path) as f:
        abi = f.read()
    
    return w3.eth.contract(address=address, abi=abi)

def to_price(sqrtPriceX96):
    return (sqrtPriceX96 / (2 ** 96)) ** 2

In [4]:
DENMANAGERGEETER = '0xFA7908287c1f1B256831c812c7194cb95BB440e6'

den_manager_getter = create_contract_web3(DENMANAGERGEETER, "abis/BeraborrowDenManagerGetters.abi")
collaterals_info = den_manager_getter.functions.getAllCollateralsAndDenManagers().call()

collaterals = []
vault_tokens_addresses = []
tokens_address = []
den_managers = []

for col in range(len(collaterals_info)):
    vault_token_address = collaterals_info[col][0]
    den_manager = collaterals_info[col][1][0]
    vault_token = create_contract_web3(vault_token_address, "abis/ERC-4626.abi")
    token_address = vault_token.functions.asset().call()
    token = create_contract_web3(token_address, "abis/WETH.abi")
    
    collaterals.append(token.functions.symbol().call())
    vault_tokens_addresses.append(vault_token_address)
    tokens_address.append(token_address)
    den_managers.append(den_manager)

collaterals_data = pd.DataFrame({
    'collateral': collaterals,
    'vault': vault_tokens_addresses,
    'token': tokens_address,
    'den_manager': den_managers
})

display(collaterals_data)

,collateral,vault,token,den_manager
0,pumpBTC.bera,0xCE1e426E35eBc9f512944F59527304E3B771EA12,0x1fCca65fb6Ae3b2758b9b2B394CB227eAE404e1E,0xFec9F8cd6F7B7a6674CD846938Fe44b001f7a568
1,SolvBTC,0x4c0c5ae255A1dcBeC206a1CD40a03c60C387dCdD,0x541FD749419CA806a8bc7da8ac23D346f2dF8B77,0xb6Ab580B36ed2DEF692dc608aFAc0A27C725CD0E
2,xSolvBTC,0xE357B38F9871da3F8320E132e272D93721aA6582,0xCC0966D8418d412c599A6421b760a847eB169A8c,0xcEDEFfec2860409E3ccB0D798C71dD1239259d8A
3,uniBTC,0xdc8408870f77B0B99d70779f68Fb560b6FE39259,0xC3827A4BC8224ee2D116637023b124CED6db6e90,0xE66E9873E5851b25a3b539879265F554c50f2f8c
4,beraETH,0x65af6D1822f162b2b70395428CC24FF5587c0B02,0x6fc6545d5cDE268D5C7f1e476D444F39c995120d,0xA0f32354419c2e02cd5D535edd5Fd7F5e81Fbe96
5,STONE,0x3D7F3b0aB008a04582A2A37bEe150dE7c8a37953,0xEc901DA9c68E90798BbBb74c11406A32A70652C3,0x2016885fC46E879Aa3D170C7472Ce623090D4c1a
6,WETH,0x42ebe55B64Bb6fA169518ab1Bf1c7fCa874e3004,0x2F6F07CDcf3588944Bf4C42aC74ff24bF56e7590,0x9A3549ef882584a687C1FF1843e3B3C07a2A0cB2
7,ylstETH,0x45b3B6b28c77DD23A91AcC3bd41dA436FA6A61AE,0xa958090601E21A82e9873042652e35891D945a8C,0xc3Ad34f9fe5C4634C9e723682a0FEdB5103cd667
8,rsETH,0xabBF8064047a6EA85169230bf42013dA4431659A,0x4186BFC76E2E237523CBC30FD220FE055156b41F,0xC94CD2ADB51E5CbE2d9fa0d09481a24929d7B7fF
9,KODI WBTC-HONEY,0x08FEDDCBF7fF3462A61Dd56016f5297B87c0789F,0xf6b16E73d3b0e2784AAe8C4cd06099BE65d092Bf,0xEe49c5FF02b70c4099EA63B1792d963ABD9947f0


In [5]:
BLOCK_RATE = 2
OBSERVATION_PERIOD = 4 * 30 * 24 * 60 * 60
LAST_BLOCK_NUMBER = w3.eth.get_block_number()
#LAST_BLOCK_NUMBER = 7355900
FIRST_BLOCK_NUMBER = LAST_BLOCK_NUMBER - int(OBSERVATION_PERIOD/BLOCK_RATE)
N = 10000

In [6]:
def getPrice(address):
    pool_address = Web3.to_checksum_address(address)
    pool = create_contract_web3(pool_address, 'abis/KodiakV3Pool.abi')

    token0_address = pool.functions.token0.call()
    token1_address = pool.functions.token1.call()
    token0 = create_contract_web3(token0_address, 'abis/WETH.abi')
    token1 = create_contract_web3(token1_address, 'abis/WETH.abi')
    scale0 = token0.functions.decimals.call()
    scale1 = token1.functions.decimals.call()

    print([token0.functions.symbol.call(), token1.functions.symbol.call()])

    timestamps = []
    prices = []

    for blocknumber in np.linspace(FIRST_BLOCK_NUMBER, LAST_BLOCK_NUMBER, N):
        timestamp = w3.eth.get_block(block_identifier=int(blocknumber))['timestamp']
        timestamps.append(timestamp)

        price_x96 = pool.functions.slot0.call(block_identifier=int(blocknumber))[0] 
        price = to_price(price_x96)*10**(scale0 - scale1)
        prices.append(price)

        time.sleep(1)

    time.sleep(10)
    return np.array(timestamps), np.array(prices)

In [7]:
NECT_HONEY_POOL = '0x56a5AC11BA0f2C6b887d3E3aA154e3dd9859eac2'
WBTC_HONEY_POOL = '0x545Bea6Ea7F8fD8dCC5C9A6802a8ebF3DbFc1C6E'

timestamp, nect_honey_price = getPrice(NECT_HONEY_POOL)

df = pd.DataFrame({
    'timestamp': timestamp,
    'price': nect_honey_price
})

df.to_csv('datas/nect_honey_price.csv', index=False)

timestamp, wbtc_honey_price = getPrice(WBTC_HONEY_POOL)

df = pd.DataFrame({
    'timestamp': timestamp,
    'price': wbtc_honey_price
})

df.to_csv('datas/wbtc_honey_price.csv', index=False)

['NECT', 'HONEY']
['WBTC', 'HONEY']


In [8]:
def getIslandData(address):
    island_address = Web3.to_checksum_address(address)
    island = create_contract_web3(island_address, 'abis/KodiakIsland.abi')
    
    token0_address = island.functions.token0.call()
    token1_address = island.functions.token1.call()
    token0 = create_contract_web3(token0_address, 'abis/WETH.abi')
    token1 = create_contract_web3(token1_address, 'abis/WETH.abi')
    scale0 = 10**token0.functions.decimals.call()
    scale1 = 10**token1.functions.decimals.call()
    scale = 10**island.functions.decimals.call()

    print([token0.functions.symbol.call(), token1.functions.symbol.call()])

    timestamps = []
    underlying_assets0 = []
    underlying_assets1 = []
    total_supplies = []

    for blocknumber in np.linspace(FIRST_BLOCK_NUMBER, LAST_BLOCK_NUMBER, N):
        timestamp = w3.eth.get_block(block_identifier=int(blocknumber))['timestamp']
        timestamps.append(timestamp)

        underlying_asset = island.functions.getUnderlyingBalances().call(block_identifier = int(blocknumber))
        underlying_assets0.append(underlying_asset[0]/scale0)
        underlying_assets1.append(underlying_asset[1]/scale1)

        totla_supply = island.functions.totalSupply().call(block_identifier = int(blocknumber))
        total_supplies.append(totla_supply/scale)

        time.sleep(1)

    time.sleep(10)

    return np.array(timestamps), np.array(underlying_assets0), np.array(underlying_assets1), np.array(total_supplies)

In [9]:
WBTC_HONEY_ISLAND = '0xf6b16e73d3b0e2784aae8c4cd06099be65d092bf'

timestamp, wbtc, honey, lp = getIslandData(WBTC_HONEY_ISLAND)

nav = (wbtc*wbtc_honey_price + honey)/lp
wbtc_frac = wbtc*wbtc_honey_price/(wbtc*wbtc_honey_price + honey)

df = pd.DataFrame({
    'timestamp': timestamp,
    'nav': nav,
    'wbtc_frac': wbtc_frac
})

df.to_csv('datas/island_data.csv', index=False)

['WBTC', 'HONEY']
